In [0]:
-- ============================================================
-- GOLD FACT TABLE: fact_marketing_performance
-- PURPOSE:
--   Store session-level marketing and behavioral performance
--   metrics in a star-schema-friendly fact table.
-- GRAIN:
--   1 row per session_id
--
-- WHY SESSION GRAIN:
--   Sessions act as the natural behavioral unit that connects
--   traffic source, device, category navigation, and conversion
--   outcomes without inflating metrics.
--
-- RELATIONSHIPS:
--   fact_marketing_performance.campaign_id
--     -> dim_campaign.campaign_id
--
--   fact_marketing_performance.channel_id
--     -> dim_channel.channel_id
--
--   fact_marketing_performance.device_id
--     -> dim_device.device_id
--
--   fact_marketing_performance.category_id
--     -> dim_category.category_id
--
--   fact_marketing_performance.customer_id
--     -> dim_customer.customer_id
--
-- ============================================================

USE CATALOG shoply_analytics;
USE SCHEMA gold;

CREATE OR REPLACE VIEW fact_marketing_performance AS
WITH

-- ============================================================
-- A) SESSION CATEGORY
--   Assign one category per session using the first category
--   page viewed during the session.
-- ============================================================
session_category_ranked AS (
    SELECT
        session_id,
        LOWER(TRIM(REGEXP_EXTRACT(page_url, '^/category/([^/?#]+)', 1))) AS category_name,
        ROW_NUMBER() OVER (
            PARTITION BY session_id
            ORDER BY timestamp ASC
        ) AS rn
    FROM shoply_analytics.silver.page_views
    WHERE page_url LIKE '/category/%'
),

session_category_one AS (
    SELECT
        session_id,
        COALESCE(NULLIF(category_name, ''), 'unknown') AS category_name
    FROM session_category_ranked
    WHERE rn = 1
),

-- ============================================================
-- B) PAGE VIEW COUNTS PER SESSION
-- ============================================================
pageview_metrics AS (
    SELECT
        session_id,
        COUNT(*) AS page_views
    FROM shoply_analytics.silver.page_views
    GROUP BY session_id
),

-- ============================================================
-- C) PURCHASE METRICS PER SESSION
--   purchase_events = number of purchase conversion rows
--   purchase_revenue = sum of purchase amount
-- ============================================================
purchase_metrics AS (
    SELECT
        session_id,
        COUNT(*) AS purchase_events,
        SUM(COALESCE(amount, 0)) AS purchase_revenue
    FROM shoply_analytics.silver.conversion
    WHERE LOWER(TRIM(conversion_type)) = 'purchase'
    GROUP BY session_id
),

-- ============================================================
-- D) SESSION BASE WITH NATURAL DIMENSION VALUES
-- ============================================================
session_base AS (
    SELECT
        s.session_id,
        DATE(s.session_start) AS date,
        s.campaign_id,
        s.user_id AS customer_id,

        COALESCE(NULLIF(s.utm_source, ''), 'unknown') AS utm_source,
        COALESCE(NULLIF(s.utm_medium, ''), 'unknown') AS utm_medium,
        COALESCE(NULLIF(s.utm_campaign, ''), 'unknown') AS utm_campaign,

        COALESCE(NULLIF(s.device_type, ''), 'unknown') AS device_type,
        COALESCE(sc.category_name, 'unknown') AS category_name,

        1 AS sessions
    FROM shoply_analytics.silver.sessions s
    LEFT JOIN session_category_one sc
        ON s.session_id = sc.session_id
)

-- ============================================================
-- E) FINAL FACT TABLE
--   Resolve surrogate dimension keys and attach session measures.
-- ============================================================
SELECT
    sb.session_id,
    sb.date,

    -- Foreign keys to dimensions
    sb.campaign_id,
    ch.channel_id,
    d.device_id,
    cat.category_id,
    sb.customer_id,

    -- Measures
    sb.sessions,
    1 AS unique_users,  -- session grain, user is represented by customer_id
    COALESCE(pv.page_views, 0) AS page_views,
    COALESCE(pm.purchase_events, 0) AS purchase_events,
    COALESCE(pm.purchase_revenue, 0) AS purchase_revenue

FROM session_base sb

LEFT JOIN gold.dim_channel ch
    ON sb.utm_source = ch.utm_source
   AND sb.utm_medium = ch.utm_medium
   AND sb.utm_campaign = ch.utm_campaign

LEFT JOIN gold.dim_device d
    ON sb.device_type = d.device_type

LEFT JOIN gold.dim_category cat
    ON sb.category_name = cat.category_name

LEFT JOIN pageview_metrics pv
    ON sb.session_id = pv.session_id

LEFT JOIN purchase_metrics pm
    ON sb.session_id = pm.session_id;